# 🛡️ Automated Violence Detection in Surveillance Systems using YOLOv8
### Google Colab Training & Evaluation Pipeline

**Reference Repository:** [aatansen/Violence-Detection-Using-YOLOv8-Towards-Automated-Video-Surveillance-and-Public-Safety](https://github.com/aatansen/Violence-Detection-Using-YOLOv8-Towards-Automated-Video-Surveillance-and-Public-Safety)

#### Dataset Specifications:
- **Source:** [Roboflow Violence Dataset (shah-xxxqs/violence-3h8pw)](https://universe.roboflow.com/shah-xxxqs/violence-3h8pw)
- **Total Images:** 2,834 images
- **Train Set:** 1,969 images (70%)
- **Validation Set:** 575 images (20%)
- **Test Set:** 290 images (10%)
- **Classes:** `0: Non-Violence`, `1: Violence`

## 1. Environment Setup & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib pandas seaborn

In [ ]:
import torch
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import cv2

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Dataset Download & Configuration

In [ ]:
# Download dataset via Roboflow API (replace YOUR_API_KEY with your key)
from roboflow import Roboflow
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

if ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("shah-xxxqs").project("violence-3h8pw")
    dataset = project.version(1).download("yolov8")
    data_yaml_path = dataset.location + "/data.yaml"
else:
    print("Using default data.yaml path setup...")
    data_yaml_path = "data.yaml"

## 3. Train YOLOv8s Model for Violence Detection

**Hyperparameters:**
- Model: `yolov8s.pt`
- Epochs: `25`
- Batch Size: `16`
- Image Resolution: `640x640`
- Confidence Threshold: `0.25`

In [ ]:
model = YOLO('yolov8s.pt')

# Execute Training
results = model.train(
    data=data_yaml_path,
    epochs=25,
    batch=16,
    imgsz=640,
    lr0=0.01,
    name='yolov8s_violence_run',
    project='runs/detect',
    plots=True
)

## 4. Model Evaluation & Benchmark Metrics

In [ ]:
# Validate on Test Set
best_model_path = 'runs/detect/yolov8s_violence_run/weights/best.pt'
if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(split='test')
    print(f"mAP@50: {metrics.box.map50:.4f}")
    print(f"mAP@50-95: {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")
else:
    print("Trained weights file not found. Ensure training completed.")

## 5. Inference on Video Surveillance Streams

In [ ]:
# Run inference on sample surveillance video
input_video = "sample_surveillance.mp4"
if os.path.exists(input_video):
    results = trained_model.predict(
        source=input_video,
        conf=0.25,
        save=True,
        project='runs/predict'
    )
    print("Video prediction saved to runs/predict/")